In [ ]:
# Parameters — edit these before running
REPO_URL    = "https://github.com/Dev252001/ham10000-lesion-eval.git"
REPO_BRANCH = "main"  # set to another branch to test before merging
SEED        = 42
SPLIT_MODE  = "lesion"  # "image" is for the leakage comparison

In [ ]:
# Setup: mount Drive, clone / checkout / pull repo, install extras
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, subprocess, sys
from pathlib import Path

DRIVE_DIR  = Path("/content/drive/MyDrive/ham10000_project")
REPO_DIR   = DRIVE_DIR / "ham10000-lesion-eval"
CKPT_DIR   = DRIVE_DIR / "checkpoints"
DATA_DIR   = Path("/content/ham10000_data")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

sys.path.insert(0, str(REPO_DIR / "src"))

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "pandas", "scikit-learn", "matplotlib", "Pillow",
     "grad-cam", "kaggle"],
    check=True,
)

In [ ]:
# Copy dataset from Drive cache to /content
import shutil

DATA_CACHE = DRIVE_DIR / "ham10000_raw"
DATA_DIR   = Path("/content/ham10000_data")
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
shutil.copytree(str(DATA_CACHE), str(DATA_DIR))
print(f"Data at {DATA_DIR}")

In [ ]:
# Build split in session and load or train baseline CNN
import torch
from skinlesion.data import load_metadata, HAMDataset
from skinlesion.splits import make_split, split_manifest, manifest_sha256
from skinlesion.transforms import train_transform, eval_transform
from skinlesion.models import build_model, compute_class_weights
from skinlesion.config import TrainConfig
from skinlesion.train import load_or_train
from torch.utils.data import DataLoader

df = load_metadata(DATA_DIR)

cfg = TrainConfig(model_name="baseline_cnn")

train_df, val_df, test_df = make_split(df, split_mode=SPLIT_MODE, seed=SEED)
manifest = split_manifest(train_df, val_df, test_df)
msha = manifest_sha256(manifest)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

train_ds = HAMDataset(train_df, train_transform(cfg.image_size))
val_ds   = HAMDataset(val_df, eval_transform(cfg.image_size))

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=cfg.batch_size, shuffle=False, num_workers=0)

class_weights = compute_class_weights(train_df)
model = build_model("baseline_cnn", pretrained=False)

ckpt_path = CKPT_DIR / f"baseline_cnn_{SPLIT_MODE}_seed{SEED}_best.pt"
history, best_epoch, epochs_run, train_time_s = load_or_train(
    model=model,
    model_name="baseline_cnn",
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    class_weights=class_weights,
    device=device,
    ckpt_path=ckpt_path,
    split_mode=SPLIT_MODE,
    seed=SEED,
    manifest_sha256=msha,
)
print(f"best_epoch={best_epoch}  epochs_run={epochs_run}  train_time={train_time_s:.1f}s")

In [ ]:
# Final test evaluation — runs ONCE on the test split
from skinlesion.data import HAMDataset
from skinlesion.transforms import eval_transform
from skinlesion.evaluate import final_test_evaluation
from torch.utils.data import DataLoader

test_ds     = HAMDataset(test_df, eval_transform(cfg.image_size))
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=0)

metrics = final_test_evaluation(
    model=model,
    model_name="baseline_cnn",
    test_df=test_df,
    test_loader=test_loader,
    cfg=cfg,
    split_mode=SPLIT_MODE,
    seed=SEED,
    manifest_sha256=msha,
    repo_dir=REPO_DIR,
    train_time_s=train_time_s,
    epochs_run=epochs_run,
    best_epoch=best_epoch,
    device=device,
    outputs_dir=REPO_DIR / "outputs",
)
print(f"Macro-F1: {metrics['metrics']['macro_f1']:.4f}")

In [ ]:
# Confusion matrix figure
import matplotlib.pyplot as plt
import numpy as np
from skinlesion.data import CLASSES

cm = metrics["metrics"]["confusion_matrix"]
cm_arr = np.array(cm)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm_arr, cmap="Blues")
ax.set_xticks(range(len(CLASSES)))
ax.set_yticks(range(len(CLASSES)))
ax.set_xticklabels(CLASSES, rotation=45, ha="right")
ax.set_yticklabels(CLASSES)
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Confusion matrix — baseline CNN")
plt.colorbar(im, ax=ax)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, str(cm_arr[i, j]), ha="center", va="center", fontsize=8)
fig.tight_layout()
cm_fig_path = REPO_DIR / "outputs" / "figures" / "confusion_matrix_baseline_cnn.png"
fig.savefig(cm_fig_path, dpi=100)
plt.close(fig)
print(f"Saved {cm_fig_path}")

In [ ]:
# Package outputs and download
from skinlesion.export import make_outputs_zip

zip_path = DRIVE_DIR / "outputs_02.zip"
make_outputs_zip(REPO_DIR, zip_path)
from google.colab import files
files.download(str(zip_path))
print("Unzip at the root of your local clone, then commit.")